In [1]:
%load_ext autoreload
%autoreload 2
%cd /mnt/sdd1/atharvas/formulacode/datasmith
import datetime

import pandas as pd

from datasmith.logging_config import get_logger

logger = get_logger("notebooks.building_reports_pr")

curr_date: str = datetime.datetime.now().isoformat()

/mnt/sdd1/atharvas/formulacode/datasmith


In [2]:
commits_df = pd.read_parquet("scratch/artifacts/pipeflush/less_filtered_commits_perfonly.parquet")

In [3]:
from datasmith.execution.collect_commits import collect_merge_shas

commits = collect_merge_shas(repo="pydata/xarray")

/mnt/sdd1/atharvas/formulacode/datasmith/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
19:43:47 WARNING  simple_useragent.core: Falling back to historic user agent.
Paginating GitHub:  23%|██▎       | 23/100 [00:00<00:00, 142.66page/s]
19:43:48 INFO     datasmith: Collected 2130 merged PR SHAs (non-null) from pydata/xarray.


In [4]:
from tqdm.auto import tqdm

from datasmith.scrape.build_pr_report import build_pr_report

reports = []


# with ThreadPoolExecutor(max_workers=100) as executor:
#     futures = {
#         executor.submit(
#             build_pr_report,
#             link=commit['url'],
#             summarize_llm=False,
#             add_classification=False,
#         ): commit for commit in commits[:750]
#     }
#     for future in tqdm(as_completed(futures), total=len(futures)):
#         commit = futures[future]
#         try:
#             report = future.result()
#             reports.append(report)
#         except Exception as e:
#             logger.error(f"Error processing commit {commit['url']}: {e}")
# above as a for loop:

for commit in tqdm(commits[:100]):
    if not commit:
        reports.append(None)
        continue
    report = build_pr_report(
        link=commit["url"],
        summarize_llm=False,
        add_classification=False,
    )
    reports.append(report)

100%|██████████| 100/100 [00:41<00:00,  2.41it/s]


In [ ]:
df = pd.DataFrame([{**c, **{"report": report}} for c, report in zip(commits, reports)])

In [ ]:
df[["url", "report"]][~df["report"].str.contains("NOT_A_VALID_PR")].url.values

array(['https://api.github.com/repos/pydata/xarray/pulls/10838',
       'https://api.github.com/repos/pydata/xarray/pulls/10741',
       'https://api.github.com/repos/pydata/xarray/pulls/10790',
       'https://api.github.com/repos/pydata/xarray/pulls/10726',
       'https://api.github.com/repos/pydata/xarray/pulls/10828',
       'https://api.github.com/repos/pydata/xarray/pulls/10778',
       'https://api.github.com/repos/pydata/xarray/pulls/10817',
       'https://api.github.com/repos/pydata/xarray/pulls/10815',
       'https://api.github.com/repos/pydata/xarray/pulls/10809',
       'https://api.github.com/repos/pydata/xarray/pulls/10755',
       'https://api.github.com/repos/pydata/xarray/pulls/10757',
       'https://api.github.com/repos/pydata/xarray/pulls/10792',
       'https://api.github.com/repos/pydata/xarray/pulls/10671',
       'https://api.github.com/repos/pydata/xarray/pulls/10658',
       'https://api.github.com/repos/pydata/xarray/pulls/10770',
       'https://api.githu

In [10]:
# logger.info(f"PR Report:\n{report}")
print(report)

NOT_A_VALID_PR
